# Chương 3 — Demo Hai Ứng Dụng `negFIN` (Google Colab — Pure Julia)

Phiên bản dành riêng cho **Google Colab (Julia Runtime)** — tự động `git clone` repository, tải dữ liệu từ **Hugging Face** (`stephent9/Lab1_Frequent_Itemset_Mining`), chạy kiểm thử Chương 3 và thực thi đầy đủ cả **2 ứng dụng thực tế thuần Julia** (*Market Basket Analysis* & *Bag of Visual Words*) bằng chính thuật toán `negFIN` tự viết của nhóm.

### Nguồn dữ liệu (Hugging Face — Public, không cần xác thực)
- **Repository:** https://huggingface.co/datasets/stephent9/Lab1_Frequent_Itemset_Mining (`groceries.csv`, `digits8x8.csv`)

### Hướng dẫn chạy trên Google Colab
1. Vào menu **Runtime $\rightarrow$ Change runtime type $\rightarrow$ chọn `Julia`** $\rightarrow$ bấm **Save**.
2. (Tùy chọn) Chỉnh các biến trong khối **SETUP & CONFIG** ở Phần 1 (`ALGO`, `MARKET_MINSUP`, `MARKET_MINCONF`, `VISUAL_MINSUP`, `VISUAL_WORDS`, `VISUAL_SEED`).
3. Bấm **Runtime $\rightarrow$ Run all** để chạy kiểm thử và xem đầy đủ kết quả + biểu đồ SVG của cả 2 ứng dụng.
4. Để chia sẻ nộp bài: **File $\rightarrow$ Save a copy in Drive** $\rightarrow$ **Share $\rightarrow$ Anyone with the link (Viewer)**.

### Mục lục các phần trong Notebook
- **Phần 1:** Thiết lập môi trường (`SETUP & CONFIG`), tự động `git clone` và nạp module `Applications`.
- **Phần 2:** Chạy bộ kiểm thử tự động Chương 3 (`tests/test_ch3_julia.jl`).
- **Phần 3 — Ứng dụng 1:** Phân tích giỏ hàng (`Groceries`) & Top-10 luật theo `Lift`.
- **Phần 4 — Ứng dụng 2:** Bag of Visual Words (`UCI Digits 8×8`) & đánh giá *Nearest-Centroid*.

---
## Phần 1: Thiết Lập Môi Trường & Nạp Module `Applications`

Cell này tự động xác định thư mục dự án (hoặc `git clone` từ GitHub khi mở độc lập trên Colab), kích hoạt `Project.toml` và nạp `src/applications/applications.jl`.

In [ ]:
using Downloads: download
using Pkg: Pkg

# ==============================================================================
# SETUP & CONFIG (CHỈNH THAM SỐ TẠI ĐÂY NẾU CẦN)
# ==============================================================================
const GITHUB_REPO_URL = "https://github.com/Thanhtung46/18.git"
const HF_DATASET_REPO = "stephent9/Lab1_Frequent_Itemset_Mining"
const REPO_BRANCH = "main"
const ALGO = :optimized       # :optimized hoặc :baseline
const MARKET_MINSUP = 0.01    # Ngưỡng support tối thiểu cho Groceries (1%)
const MARKET_MINCONF = 0.15   # Ngưỡng confidence tối thiểu cho Groceries (15%)
const VISUAL_MINSUP = 0.12    # Ngưỡng support tối thiểu cho Digits (12%)
const VISUAL_WORDS = 16       # Số từ thị giác K trong K-means
const VISUAL_SEED = 42        # Seed cố định đảm bảo tái lập

# 1. Xác định thư mục gốc dự án (tự động git clone nếu mở độc lập trên Colab)
function resolve_project_root()::String
    dir = pwd()
    for _ in 1:4
        if isfile(joinpath(dir, "src", "applications", "applications.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && break
        dir = parent
    end
    clone_dir = joinpath(pwd(), "18")
    if !isdir(clone_dir)
        println("Đang clone mã nguồn dự án từ GitHub cho môi trường Colab...")
        run(`git clone --depth 1 --branch $REPO_BRANCH $GITHUB_REPO_URL $clone_dir`)
    end
    return clone_dir
end

const ROOT_DIR = resolve_project_root()
const CH3_DATA_DIR = joinpath(ROOT_DIR, "data", "chapter3")
mkpath(CH3_DATA_DIR)

# 2. Tải dữ liệu Chương 3 từ Hugging Face nếu chưa có
for fname in ("groceries.csv", "digits8x8.csv")
    target = joinpath(CH3_DATA_DIR, fname)
    if !isfile(target)
        hf_url = "https://huggingface.co/datasets/$HF_DATASET_REPO/resolve/main/$fname"
        println("Đang tải $fname từ Hugging Face...")
        download(hf_url, target)
    end
end

# 3. Kích hoạt môi trường & nạp module Applications
Pkg.activate(ROOT_DIR)
include(joinpath(ROOT_DIR, "src", "applications", "applications.jl"))
using .Applications: print_accuracy_table, print_top_rules_table,
    run_market_basket, run_visual_words

println("Julia version: ", VERSION)
println("Project root:  ", ROOT_DIR)

---
## Phần 2: Chạy Kiểm Thử Tự Động Chương 3 (`tests/test_ch3_julia.jl`)

In [ ]:
include(joinpath(ROOT_DIR, "tests", "test_ch3_julia.jl"))

---
## Phần 3: Ứng Dụng 1 — Phân Tích Giỏ Hàng (`Groceries`, `negFIN`)

- Dữ liệu: 9,835 giao dịch, 169 nhóm sản phẩm (tự động tải vào `data/chapter3/groceries.csv` khi chạy lần đầu).
- Lưu ý: Chỉ số `Lift > 1` phản ánh **tương quan thống kê dương**, không chứng minh quan hệ nhân quả.

In [ ]:
market_res = run_market_basket(;
    minsup=MARKET_MINSUP,
    minconf=MARKET_MINCONF,
    algo=ALGO,
)

market_dir = joinpath(ROOT_DIR, "results", "chapter3", "market_basket")
println("\n=== MARKET BASKET SUMMARY ===")
println(read(joinpath(market_dir, "summary.txt"), String))
println("=== TOP-10 LUẬT THEO LIFT ===")
print_top_rules_table(joinpath(market_dir, "top10_lift.csv"))

display("image/svg+xml", read(joinpath(market_dir, "top10_lift.svg"), String))

---
## Phần 4: Ứng Dụng 2 — Bag of Visual Words (`UCI Digits 8×8`, `negFIN`)

- Trích 9 dense gradient descriptors 6 chiều/ảnh (**không phải SIFT/ORB**), học codebook K-means trên tập train, khai thác mẫu bằng `negFIN`.
- Đánh giá độ làm giàu đặc trưng (*Enrichment*) và độ chính xác *Nearest-Centroid* trên tập test độc lập.

In [ ]:
visual_res = run_visual_words(;
    minsup=VISUAL_MINSUP,
    words=VISUAL_WORDS,
    seed=VISUAL_SEED,
    algo=ALGO,
)

visual_dir = joinpath(ROOT_DIR, "results", "chapter3", "visual_words")
println("\n=== VISUAL WORDS SUMMARY ===")
println(read(joinpath(visual_dir, "summary.txt"), String))
println("=== SO SÁNH ĐỘ CHÍNH XÁC NEAREST-CENTROID ===")
print_accuracy_table(joinpath(visual_dir, "test_accuracy_comparison.csv"))

display("image/svg+xml", read(joinpath(visual_dir, "enriched_patterns.svg"), String))
display("image/svg+xml", read(joinpath(visual_dir, "digits_samples.svg"), String))